## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    col,
    lit
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [8]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("lab 1 Test")
    conf.setMaster("local[*]")
    conf.set("spark.submit.deployMode", "client")
    #conf.set("spark.executor.memory", "12g")
    #conf.set("spark.executor.cores", "8")
    #conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "8g")
    conf.set("spark.driver.cores", "2")
    #conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    #conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    #conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    #conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    #conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    #conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

Создаём сам объект конфигурации.

In [9]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [10]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет `"US Used cars dataset"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/ananaymital/us-used-cars-dataset.

Датасет включает в себя информацию о более чем трех миллионах используемых машин в США. Он разрешен для использования в учебных целях.

Данный датасет уже загружен в `HDFS` по адресу: `hdfs:///datasets/used_cars_data.csv`

Указываем путь в `HDFS` для файла с данными.

In [18]:
path = "used_cars_data.csv"

Заполняем датафрейм данными из файла.

In [19]:
df = (spark.read.format("csv")
      .option("header", "true")
      .load(path)
)

Выводим фрагмент датафрейма на экран.

In [21]:
df.limit(10).toPandas()

,vin,back_legroom,bed,bed_height,bed_length,body_type,cabin,city,city_fuel_economy,combine_fuel_economy,...,transmission,transmission_display,trimId,trim_name,vehicle_damage_category,wheel_system,wheel_system_display,wheelbase,width,year
0,ZACNJABB5KPJ92081,35.1 in,None,None,None,SUV / Crossover,None,Bayamon,None,None,...,A,9-Speed Automatic Overdrive,t83804,Latitude FWD,None,FWD,Front-Wheel Drive,101.2 in,79.6 in,2019
1,SALCJ2FX1LH858117,38.1 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,9-Speed Automatic Overdrive,t86759,S AWD,None,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
2,JF1VA2M67G9829723,35.4 in,None,None,None,Sedan,None,Guaynabo,17.0,None,...,M,6-Speed Manual,t58994,Base,None,AWD,All-Wheel Drive,104.3 in,78.9 in,2016
3,SALRR2RV0L2433391,37.6 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,8-Speed Automatic Overdrive,t86074,V6 HSE AWD,None,AWD,All-Wheel Drive,115 in,87.4 in,2020
4,SALCJ2FXXLH862327,38.1 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,9-Speed Automatic Overdrive,t86759,S AWD,None,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
5,SALYK2EX1LA261711,37.1 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,8-Speed Automatic Overdrive,t85614,P250 R-Dynamic S AWD,None,AWD,All-Wheel Drive,113.1 in,84.4 in,2020
6,3MZBPABL6KM107908,35.1 in,None,None,None,Sedan,None,Bayamon,None,None,...,A,6-Speed Automatic Overdrive,t85256,Sedan FWD,None,FWD,Front-Wheel Drive,107.3 in,70.7 in,2019
7,SALYK2EX5LA275434,37.1 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,8-Speed Automatic Overdrive,t85614,P250 R-Dynamic S AWD,None,AWD,All-Wheel Drive,113.1 in,84.4 in,2020
8,SALCJ2FX6LH858128,38.1 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,9-Speed Automatic Overdrive,t86759,S AWD,None,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
9,SALZL2GX4LH007593,33.8 in,None,None,None,SUV / Crossover,None,San Juan,None,None,...,A,9-Speed Automatic Overdrive,t85531,P300 R-Dynamic SE AWD,None,AWD,All-Wheel Drive,105.6 in,82.7 in,2020


Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все солбцы датасета. Оставим следующие колонки, удалив остальные:

| Название столбца  | Расшифровка |
| ------------- | ------------- |
| vin               | Идентификационный номер автомобиля  |
| body_type         | Тип кузова автомобиля (кабриолет, хэтчбек, седан и т.д.)  |
| daysonmarket      | Количество дней, прошедших с момента первого размещения автомобиля на сайте |
| fleet             | Был ли автомобиль ранее частью автопарка |
| has_accidents     | Имеется ли регистрация аварий на данном автомобиле |
| horsepower        | Мощность двигателя в лошадиных силах |
| is_certified      | Является ли автомобиль сертифицированным (на сертифицированные автомобили распространяется гарантийный срок) |
| is_cpo            | Подержанные автомобили, сертифицированные дилером |
| is_oemcpo         | Подержанные автомобили, сертифицированные производителем |
| major_options     | Список основный опций автомобиля |
| maximum_seating   | Максимальное количество посадочных мест |
| mileage           | Величина пробега автомобиля |
| price             | Цена автомобиля |
| wheel_system      | Тип привода |
| year              | Год выпуска автомобиля |

In [22]:
df = df.select(
    "vin", "body_type", "daysonmarket", "fleet", "has_accidents", "horsepower",
    "is_certified", "is_cpo", "is_oemcpo", "major_options",
    "maximum_seating", "mileage", "price", "wheel_system", "year"
)

In [24]:
df.show()

+-----------------+---------------+------------+-----+-------------+----------+------------+------+---------+--------------------+---------------+-------+-------+------------+----+
|              vin|      body_type|daysonmarket|fleet|has_accidents|horsepower|is_certified|is_cpo|is_oemcpo|       major_options|maximum_seating|mileage|  price|wheel_system|year|
+-----------------+---------------+------------+-----+-------------+----------+------------+------+---------+--------------------+---------------+-------+-------+------------+----+
|ZACNJABB5KPJ92081|SUV / Crossover|         522| NULL|         NULL|     177.0|        NULL|  NULL|     NULL|['Quick Order Pac...|        5 seats|    7.0|23141.0|         FWD|2019|
|SALCJ2FX1LH858117|SUV / Crossover|         207| NULL|         NULL|     246.0|        NULL|  NULL|     NULL|['Adaptive Cruise...|        7 seats|    8.0|46500.0|         AWD|2020|
|JF1VA2M67G9829723|          Sedan|        1233|False|        False|     305.0|        NULL|  N

Выведем на экран метаданные датасета.

In [25]:
df.printSchema()

root
 |-- vin: string (nullable = true)
 |-- body_type: string (nullable = true)
 |-- daysonmarket: string (nullable = true)
 |-- fleet: string (nullable = true)
 |-- has_accidents: string (nullable = true)
 |-- horsepower: string (nullable = true)
 |-- is_certified: string (nullable = true)
 |-- is_cpo: string (nullable = true)
 |-- is_oemcpo: string (nullable = true)
 |-- major_options: string (nullable = true)
 |-- maximum_seating: string (nullable = true)
 |-- mileage: string (nullable = true)
 |-- price: string (nullable = true)
 |-- wheel_system: string (nullable = true)
 |-- year: string (nullable = true)



Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [27]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных и
    выполняет необходимые преобразования.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    # Преобразуем столбцы в соответствующие типы данных
    data = data.withColumn("daysonmarket",
                           col("daysonmarket").cast("Integer"))
    data = data.withColumn("fleet",
                           col("fleet").cast("Boolean"))
    data = data.withColumn("has_accidents",
                           col("has_accidents").cast("Boolean"))
    data = data.withColumn("horsepower",
                           col("horsepower").cast("Float"))
    data = data.withColumn("is_certified",
                           col("is_certified").cast("Boolean"))
    data = data.withColumn("is_cpo",
                           col("is_cpo").cast("Boolean"))
    data = data.withColumn("is_oemcpo",
                           col("is_oemcpo").cast("Boolean"))

    # Преобразуем строку в массив строк
    data = data.withColumn("major_options",
                           regexp_extract_all(col("major_options"),
                                              lit(r"'([^']*)'"),
                                              1)
    )

    # Убираем единицы измерения
    data = data.withColumn("maximum_seating",
                           regexp_replace(col("maximum_seating"),
                                          r"\s+seats",
                                          "").cast("Integer")
    )

    data = data.withColumn("mileage",
                           col("mileage").cast("Float"))
    data = data.withColumn("price",
                           col("price").cast("Float"))
    data = data.withColumn("year",
                           col("year").cast("Integer"))

    return data

In [28]:
df = transform_dataframe(df)

In [29]:
df.show()

+-----------------+---------------+------------+-----+-------------+----------+------------+------+---------+--------------------+---------------+-------+-------+------------+----+
|              vin|      body_type|daysonmarket|fleet|has_accidents|horsepower|is_certified|is_cpo|is_oemcpo|       major_options|maximum_seating|mileage|  price|wheel_system|year|
+-----------------+---------------+------------+-----+-------------+----------+------------+------+---------+--------------------+---------------+-------+-------+------------+----+
|ZACNJABB5KPJ92081|SUV / Crossover|         522| NULL|         NULL|     177.0|        NULL|  NULL|     NULL|[Quick Order Pack...|              5|    7.0|23141.0|         FWD|2019|
|SALCJ2FX1LH858117|SUV / Crossover|         207| NULL|         NULL|     246.0|        NULL|  NULL|     NULL|[Adaptive Cruise ...|              7|    8.0|46500.0|         AWD|2020|
|JF1VA2M67G9829723|          Sedan|        1233|false|        false|     305.0|        NULL|  N

In [30]:
df.printSchema()

root
 |-- vin: string (nullable = true)
 |-- body_type: string (nullable = true)
 |-- daysonmarket: integer (nullable = true)
 |-- fleet: boolean (nullable = true)
 |-- has_accidents: boolean (nullable = true)
 |-- horsepower: float (nullable = true)
 |-- is_certified: boolean (nullable = true)
 |-- is_cpo: boolean (nullable = true)
 |-- is_oemcpo: boolean (nullable = true)
 |-- major_options: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- maximum_seating: integer (nullable = true)
 |-- mileage: float (nullable = true)
 |-- price: float (nullable = true)
 |-- wheel_system: string (nullable = true)
 |-- year: integer (nullable = true)



Видно, что теперь столбцы датафрейма содержат значения корректных типов.

Полученный датафрейм сохраним для дальнейшего использования. Сохранение выполним в таблицу `Apache Iceberg`. 

`Apache Iceberg` — это поддерживающий высокую производительность табличный формат для больших данных.

Сначала создадим базу данных, в которой будет расположена таблица. Во избежание путаницы, **каждую таблицу следует называть с использованием фамилии студента**.

In [31]:
database_name = "dolgova_database"

Создадим инструкцию SQL для добавления базы данных в каталог `Apache Spark`.

In [32]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [33]:
spark.sql(create_database_sql)

DataFrame[]

Установим созданную базу данных как текущую.

In [34]:
spark.catalog.setCurrentDatabase(database_name)

И, наконец, записываем преобразованный датафрейм в таблицу `sobd_lab1_table`.

In [36]:
# Сохранение DataFrame в виде таблицы
df.writeTo("sobd_lab1_table").using("parquet").create()

После успешной записи можно посмотреть, какие таблицы входят в базу данных.

In [37]:
for table in spark.catalog.listTables():
    print(table.name)

sobd_lab1_table


Обратите внимание, что при необходимости созданные базу данных и таблицу можно удалить следующими командами.

In [20]:
# spark.sql("DROP TABLE spark_catalog.ivanov_database.sobd_lab1_table")
# spark.sql("DROP DATABASE spark_catalog.ivanov_database")

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [38]:
spark.stop()